In [75]:
# 导入所需的库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import matplotlib.cm as cm
import lightgbm as lgb
from lightgbm import LGBMRegressor
from scipy import *  
import random
from sklearn.metrics import mean_squared_error,r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GridSearchCV
from tabpfn import TabPFNRegressor
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from typing import Any, Literal, NamedTuple, Optional
import shap

In [76]:
# 定义评价指标
def evaluate_regress(y_pre, y_true,y_name=''):

    print('*****************************************************')
   
    MAE=np.sum(np.abs(y_pre-y_true))/len(y_true)
    print(y_name+'MAE为: ',str(MAE))

    MCE=np.sum(np.abs((y_pre-y_true)/max(y_true)))/len(y_true)
    print(y_name+'MAPE为: ',str(MCE))

    MSE=np.sum((y_pre-y_true) ** 2)/len(y_true)
    print(y_name+'MSE为: ',str(MSE))
    
    RMSE=np.sqrt(MSE)
    print(y_name+'RMSE为: ',str(RMSE))

    R2=r2_score(y_true, y_pre)
    print(y_name+'R2为: ',str(R2))

    print('*****************************************************')

    return MAE,MCE,MSE,RMSE,R2

In [77]:
# 多模型输入 + GridSearchCV 自动寻优
def train_muti_model(X_train, y_train, X_test, y_test, method_choose):
    
    # 通用 GridSearch 配置
    # n_jobs=-1 表示使用所有 CPU 核心加速
    # cv=5 表示 5 折交叉验证
    # scoring='r2' 表示以 R2 分数作为评价标准
    
    if method_choose == 'LightGBM':
        # 注意：这里必须用 LGBMRegressor 而不是 lgb.train
        estimator = LGBMRegressor(objective='regression', force_col_wise=True, verbose=-1, random_state=42)
        param_grid = {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1, 0.2],
            'num_leaves': [15, 31],      # 控制复杂度
            'max_depth': [3, 5, -1]      # 限制深度防过拟合
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'XGBoost':
        estimator = XGBRegressor(random_state=42, n_jobs=-1)
        param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 7],       # 小样本建议浅一点
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.8, 1.0]       # 采样比例
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'CatBoost':
        # CatBoost 自身集成度很高，GridSearch 比较慢，这里设置简单的搜索
        estimator = CatBoostRegressor(random_seed=42, verbose=0, allow_writing_files=False)
        param_grid = {
            'iterations': [100, 200],
            'learning_rate': [0.01, 0.1],
            'depth': [4, 6]
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'RF':
        estimator = RandomForestRegressor(random_state=42)
        param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5]
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'MLP':
        estimator = MLPRegressor(random_state=42, max_iter=1000) # 增加迭代次数保证收敛
        param_grid = {
            'hidden_layer_sizes': [(32,), (64,), (64, 32)],
            'activation': ['relu', 'tanh'],
            'solver': ['adam'],
            'alpha': [0.0001, 0.01] # 正则化项
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'DT':
        estimator = DecisionTreeRegressor(random_state=42)
        param_grid = {
            'max_depth': [3, 5, 10, None],
            'min_samples_split': [2, 5, 10]
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'Bay':
        # 贝叶斯岭回归参数较少，主要调正则化参数
        estimator = BayesianRidge()
        param_grid = {
            'alpha_1': [1e-6, 1e-5],
            'alpha_2': [1e-6, 1e-5],
            'lambda_1': [1e-6, 1e-5],
            'lambda_2': [1e-6, 1e-5]
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'KNN':
        estimator = KNeighborsRegressor()
        param_grid = {
            'n_neighbors': [3, 5, 7, 9], # 邻居数量
            'weights': ['uniform', 'distance'],
            'p': [1, 2] # 1=曼哈顿距离, 2=欧氏距离
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'GBR':
        estimator = GradientBoostingRegressor(random_state=42)
        param_grid = {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.1],
            'max_depth': [3, 5]
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

    elif method_choose == 'TabPFN':
        #TabPFN 是预训练模型，通常不需要也不建议做 GridSearch
        # 直接使用默认参数即可
        print("TabPFN 使用预训练模式，跳过 GridSearch...")
        best_model = TabPFNRegressor(device='cpu') 
        best_model.fit(X_train, y_train)

    elif method_choose == 'LinearRegression':
        estimator = LinearRegression()
        param_grid = {
            'fit_intercept': [True, False],
            'positive': [False, True]
        }
        grid = GridSearchCV(estimator, param_grid, cv=10, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_


    # ------------------- 预测部分 -------------------
    # 使用找到的最佳模型进行预测
    y_test_predict = best_model.predict(X_test)
    y_train_predict = best_model.predict(X_train)
    return y_train_predict, y_test_predict, best_model

In [78]:
def test_method(y_train_predict, y_test_predict, train_y, test_y, method_label):
    train_ev = []
    test_ev = []
    
    # 定义输出时的标签名称
    method_label1 = method_label + '_训练集_输出 '
    method_label3 = method_label + '_测试集_输出 '
    
    # =================== 核心修改 ===================
    # 使用 np.array() 将输入强制转换为 numpy 数组
    # 这样无论输入是 Series 还是 Array，都有 .flatten() 方法了
    y_train_pred_flat = np.array(y_train_predict).flatten()
    y_train_true_flat = np.array(train_y).flatten()
    
    y_test_pred_flat = np.array(y_test_predict).flatten()
    y_test_true_flat = np.array(test_y).flatten()
    # ===============================================

    # 1. 评估训练集
    MAE_train, MCE_train, MSE_train, RMSE_train, R2_train = evaluate_regress(
        y_train_pred_flat, y_train_true_flat, y_name=method_label1
    )
    
    # 2. 评估测试集
    MAE_test, MCE_test, MSE_test, RMSE_test, R2_test = evaluate_regress(
        y_test_pred_flat, y_test_true_flat, y_name=method_label3
    )
    
    # 将结果存入列表
    train_ev.append([MAE_train, MCE_train, MSE_train, RMSE_train, R2_train])
    test_ev.append([MAE_test, MCE_test, MSE_test, RMSE_test, R2_test])
    
    return train_ev, test_ev

In [79]:
# 1. 读取数据
df_all = pd.read_csv(r'E:\desk_top\深圳市数据分类\Processed_Data\LCZ4-2.csv')  
df_process = df_all.dropna()

X = df_process.iloc[:, :-1] 
y = df_process.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
method_label = 'LinearRegression'

# 调用修改后的 train_muti_model
y_train_predict, y_test_predict, model = train_muti_model(
    X_train, y_train,
    X_test, y_test,
    method_label
)

print("\n--- 训练集评估 ---")
evaluate_regress(y_train_predict, y_train.values, y_name=method_label + '_训练集')

print("\n--- 测试集评估 ---")
evaluate_regress(y_test_predict, y_test.values, y_name=method_label + '_测试集')

print("\n✅ 流程执行完毕！")


--- 训练集评估 ---
*****************************************************
LinearRegression_训练集MAE为:  3.8811894670799494
LinearRegression_训练集MAPE为:  0.28721681577469227
LinearRegression_训练集MSE为:  28.584946535148728
LinearRegression_训练集RMSE为:  5.346489178437447
LinearRegression_训练集R2为:  0.08452950042790974
*****************************************************

--- 测试集评估 ---
*****************************************************
LinearRegression_测试集MAE为:  3.58984569578159
LinearRegression_测试集MAPE为:  0.3329820048216373
LinearRegression_测试集MSE为:  16.67143792411931
LinearRegression_测试集RMSE为:  4.0830672201323495
LinearRegression_测试集R2为:  -0.20814103464872247
*****************************************************

✅ 流程执行完毕！


In [80]:
# 定义要训练的模型列表
name_label_list = ['LightGBM', 'XGBoost', 'CatBoost', 'TabPFN', 'RF','KNN', 'MLP', 'DT', 'Bay', 'GBR', 'LinearRegression']
# name_label_list = ['LightGBM', 'XGBoost', 'CatBoost', 'RF', 'KNN', 'MLP', 'DT', 'Bay', 'GBR', 'LinearRegression']

# 定义评估指标名称
evaluate_list = ['MAE', 'MAPE', 'MSE', 'RMSE', 'R2']

# 初始化存储结果的列表
train_ev_list = []
test_ev_list = []
y_train_predict_list = []  # 存储训练集预测值
y_test_predict_list = []   # 存储测试集预测值
model_list = []            # 存储训练好的模型对象

# --- 循环训练每个模型 ---
for method_label in name_label_list:
    print(f"\n====== 正在处理模型: {method_label} ======")
    
    # 1. 训练与预测
    y_train_predict, y_test_predict, model = train_muti_model(
        X_train, y_train,
        X_test, y_test,
        method_label
    )
    
    # 2. 评估结果
    train_ev, test_ev = test_method(
        y_train_predict, y_test_predict,
        y_train, y_test,
        method_label
    )
    
    # 3. 收集结果
    train_ev_list.append(train_ev)
    test_ev_list.append(test_ev)
    
    y_train_predict_list.append(y_train_predict) # 保存预测值
    y_test_predict_list.append(y_test_predict)
    model_list.append(model)

# --- 构建最终的评估 DataFrame (基于测试集) ---
# 创建一个空矩阵来存放结果：行数=模型数，列数=5(MAE, MAPE, MSE, RMSE, R2)
results_matrix = np.zeros((len(name_label_list), 5))

for i in range(len(name_label_list)):
    # test_ev_list[i] 是一个包含一个列表的列表 [[MAE, MAPE, MSE, RMSE, R2]]
    # 取 [0] 拿到里面的数值列表
    results_matrix[i, :] = test_ev_list[i][0]

# 转换为 DataFrame，行索引为模型名，列索引为指标名
y_test_out_put_Df = pd.DataFrame(
    data=results_matrix, 
    columns=evaluate_list, 
    index=name_label_list
)

# --- 5. 打印结果 DataFrame ---
print("\n✅ 所有模型训练及评估完成！")
print("\n====== 🏆 所有模型测试集最终排名 (按 R2 降序) ======")

# 按 R2 分数从高到低排序显示，这样一眼就能看出哪个模型最好
final_ranking = y_test_out_put_Df.sort_values(by='R2', ascending=False)
final_ranking


====== 正在处理模型: LightGBM ======
*****************************************************
LightGBM_训练集_输出 MAE为:  2.098453239952528
LightGBM_训练集_输出 MAPE为:  0.1552902950869613
LightGBM_训练集_输出 MSE为:  8.571739309297502
LightGBM_训练集_输出 RMSE为:  2.9277532869587053
LightGBM_训练集_输出 R2为:  0.725478777508461
*****************************************************
*****************************************************
LightGBM_测试集_输出 MAE为:  3.7593713279101943
LightGBM_测试集_输出 MAPE为:  0.34870663190551754
LightGBM_测试集_输出 MSE为:  22.81424246155979
LightGBM_测试集_输出 RMSE为:  4.776425699365561
LightGBM_测试集_输出 R2为:  -0.6532960514677204
*****************************************************

====== 正在处理模型: XGBoost ======
*****************************************************
XGBoost_训练集_输出 MAE为:  2.2845830366249116
XGBoost_训练集_输出 MAPE为:  0.1690643218317195
XGBoost_训练集_输出 MSE为:  9.866514561640543
XGBoost_训练集_输出 RMSE为:  3.141100851873518
XGBoost_训练集_输出 R2为:  0.6840118975323676
********************************************

,MAE,MAPE,MSE,RMSE,R2
TabPFN,2.887628,0.267847,10.835158,3.291680,0.214801
Bay,3.499019,0.324557,15.788278,3.973447,-0.144140
MLP,3.502293,0.324861,16.359991,4.044749,-0.185571
LinearRegression,3.589846,0.332982,16.671438,4.083067,-0.208141
XGBoost,3.570259,0.331165,17.663706,4.202821,-0.280048
KNN,3.597324,0.333676,18.696626,4.323959,-0.354902
CatBoost,3.945711,0.365991,21.421591,4.628346,-0.552374
GBR,3.884948,0.360355,22.584479,4.752313,-0.636646
LightGBM,3.759371,0.348707,22.814242,4.776426,-0.653296
RF,3.974847,0.368693,24.264777,4.925929,-0.758413


In [81]:
name_label_list = ['LightGBM', 'XGBoost', 'CatBoost', 'RF',
                   'KNN', 'MLP', 'DT', 'Bay', 'GBR', 'LinearRegression']

In [82]:
# ================= 1. 设置要融合的模型索引 =================
# 0:LightGBM, 1:XGBoost, 2:CatBoost, 3:TabPFN
combined_model_list = [0, 2]  # 这里选择了 CatBoost 和 TabPFN

# ================= 2. 计算权重 (基于测试集 MAE) =================
# 注意：test_ev_list 的结构是 [[MAE, MAPE, MSE, RMSE, R2]], 所以取 [0][0]
MAE1 = test_ev_list[combined_model_list[0]][0][0]
MAE2 = test_ev_list[combined_model_list[1]][0][0]

# 防止除以0的保护措施
if MAE1 == 0: MAE1 = 1e-6
if MAE2 == 0: MAE2 = 1e-6

# 误差倒数加权法：误差越小，权重越大
weight_com1 = (1 / MAE1) / (1 / MAE1 + 1 / MAE2)
weight_com2 = (1 / MAE2) / (1 / MAE1 + 1 / MAE2)

print(f"组合权重 -> 模型1(Index {combined_model_list[0]}): {weight_com1:.4f}, 模型2(Index {combined_model_list[1]}): {weight_com2:.4f}")

# ================= 3. 定义融合预测函数 =================
def predict_combined_shap(x):
    # 获取训练好的模型对象
    model1 = model_list[combined_model_list[0]]
    model2 = model_list[combined_model_list[1]]
    
    # 分别预测
    # 注意：这里直接使用 x (原始数据)，不需要反标准化
    predict1 = model1.predict(x).flatten()
    predict2 = model2.predict(x).flatten()
    
    # 加权求和
    predict_com = predict1 * weight_com1 + predict2 * weight_com2
    return predict_com

# ================= 4. 在测试集上进行验证 =================
# 直接传入 X_test (原始数据)
y_predict_test_com = predict_combined_shap(X_test)

# 评估结果 (传入 test_y 原始标签)
# 名字可以改一下，方便识别
new_method_name = 'Combined_CatBoost_DT'
MAE_test1, MCE_test1, MSE_test1, RMSE_test1, R2_test1 = evaluate_regress(
    y_predict_test_com, 
    y_test, 
    y_name=new_method_name
)

# ================= 5. 将结果追加到列表 =================
name_label_list.append(new_method_name)
test_ev_list.append([[MAE_test1, MCE_test1, MSE_test1, RMSE_test1, R2_test1]])

# 如果你想把新结果也打印到最后的 DataFrame 里，需要重新运行生成 DataFrame 的代码
# (即之前那个 y_test_out_put_Df 的部分)

组合权重 -> 模型1(Index 0): 0.5121, 模型2(Index 2): 0.4879
*****************************************************
Combined_CatBoost_DTMAE为:  3.8038435234681724
Combined_CatBoost_DTMAPE为:  0.35283172309066707
Combined_CatBoost_DTMSE为:  21.582367572055066
Combined_CatBoost_DTRMSE为:  4.645682680947448
Combined_CatBoost_DTR2为:  -0.5640248913951473
*****************************************************


In [ ]:
# 做可解释分析